In [25]:
import torch
import torch.nn as nn
import math
from dataclasses import dataclass

@dataclass
class ModelArgs:
    n_heads: int
    dim: int
    hidden_dim: int
    dropout: float
    max_seq_len: int
    n_layers: int

@dataclass
class FNNArgs:
    dim: int
    hidden_dim: int
    dropout: float

class MultiHeadAttention(nn.Module):
    def __init__(self, args: ModelArgs, is_causal=True):
        super().__init__()
        self.args = args
        assert args.dim % args.n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads = args.n_heads
        self.head_dim = args.dim // args.n_heads
        self.is_causal = is_causal
        self.wq = nn.Linear(args.dim, args.dim, bias=False)
        self.wk = nn.Linear(args.dim, args.dim, bias=False)
        self.wv = nn.Linear(args.dim, args.dim, bias=False)
        self.wo = nn.Linear(args.dim, args.dim, bias=False)
        # 注意力的dropout
        self.attn_dropout = nn.Dropout(args.dropout)
        
        # 残差的dropout
        self.res_dropout = nn.Dropout(args.dropout)
        if self.is_causal:
            mask = torch.full((1,1,args.max_seq_len, args.max_seq_len), float('-inf'))
            mask = torch.triu(mask, diagonal=1)
            self.register_buffer('mask', mask)


    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
        '''
        [batch_size, seq_len, dim]
        '''
        batch_size, kv_len, dim = k.shape
        _ , q_len, _ = q.shape
        Q = self.wq(q).view(batch_size, q_len, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.wk(k).view(batch_size, kv_len, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.wv(v).view(batch_size, kv_len, self.n_heads, self.head_dim).transpose(1, 2)

        atten = torch.matmul(Q, K.transpose(-2, -1)) / self.head_dim ** 0.5
        if self.is_causal : 
            atten = atten + self.mask[:, :, :q_len, :kv_len]

        score = torch.nn.functional.softmax(atten, dim=-1)

        output = self.attn_dropout(score)
        
        output = torch.matmul(score, V)

        output = output.transpose(1, 2).contiguous().view(batch_size, q_len, dim)
        
        output = self.wo(output)
        
        output = self.res_dropout(output)
        return output
        

class LayerNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        [batch_size, seq_len, dim]
        '''
        batch_size, seq_len, dim = x.shape
        x = (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + self.eps)
        x = x * self.weight + self.bias
        return x

class FNN(nn.Module):
    def __init__(self, args: FNNArgs):
        super().__init__()
        self.w1 = nn.Linear(args.dim, args.hidden_dim, bias=False)
        self.w2 = nn.Linear(args.hidden_dim, args.dim, bias=False)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(args.dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        [batch_size, seq_len, dim]
        '''
        x = self.w1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.w2(x)
        return x


@dataclass
class Args:
    max_seq_len: int
    dim: int
class PositionEncoding(nn.Module):
    def __init__(self, args: Args):
        super().__init__()
        self.pe = torch.zeros(args.max_seq_len, args.dim)
        self.position = torch.arange(0,args.max_seq_len).unsqueeze(1)
        '''
        PE(pos, 2i)   = sin( pos / 10000^(2i/d) )
        PE(pos, 2i+1) = cos( pos / 10000^(2i/d) )
        1 / 10000^(2i/d)
        = 10000^(-2i/d)
        = exp( ln(10000^(-2i/d)) )         ← 任何 a = exp(ln(a))
        = exp( (-2i/d) · ln(10000) )       ← log 性质:ln(a^b) = b·ln(a)
        = exp( 2i · (-ln(10000) / d) )     ← 把负号和除法挪进去
            ↑              ↑
        arange(0,d,2)   -math.log(10000.0)/d
        '''
        factor = torch.exp(torch.arange(0, args.dim, 2) * (-math.log(10000.0) / args.dim))

        # 计算 PE(pos, 2i)
        # 偶数部分
        self.pe[:, 0::2] = torch.sin(self.position * factor)
        # 奇数部分
        self.pe[:, 1::2] = torch.cos(self.position * factor)

        # 插入 batch 维度 [1, max_seq_len, dim]
        self.pe = self.pe.unsqueeze(0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.shape[1], :]
        return x


class EncoderLayer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.attention = MultiHeadAttention(args, is_causal=False)
        self.attention_norm = LayerNorm(args.dim)
        self.ffn_norm = LayerNorm(args.dim)
        self.ffn = FNN(FNNArgs(dim=args.dim, hidden_dim=args.hidden_dim, dropout=args.dropout))
    
    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        '''
        [batch_size, seq_len, dim]
        '''
        atten = self.attention(q, k, v)
        # 残差连接
        atten = atten + q
        atten = self.attention_norm(atten)
        # 这里的atten要保存下来，后面x要相加
        x = atten
        x = self.ffn(x)
        # 残差连接
        x = x + atten
        x = self.ffn_norm(x)
        return x

class Encoder(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layers)])
        self.position_encoding = PositionEncoding(args)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        [batch_size, seq_len, dim]
        '''
        x = self.position_encoding(x)
        for layer in self.layers:
            x = layer(x, x, x)
        return x

# Case 1

In [26]:
args = ModelArgs(n_heads=8, dim=768, hidden_dim=768*4, dropout=0.1, max_seq_len=512, n_layers=6)
print(args)

batch_size = 10

# 定义输入
x = torch.randn(batch_size, args.max_seq_len, args.dim)

# Encoder
encoder = Encoder(args)

encoder_output = encoder(x)

print("encoder output shape: ", encoder_output.shape)
print("encoder output: ", encoder_output)

ModelArgs(n_heads=8, dim=768, hidden_dim=3072, dropout=0.1, max_seq_len=512, n_layers=6)


encoder output shape:  torch.Size([10, 512, 768])
encoder output:  tensor([[[-2.1012e+00, -3.5021e-01,  8.6080e-01,  ...,  4.3152e-01,
           5.3176e-01, -8.1559e-01],
         [-3.1972e-01, -1.6823e+00, -6.3028e-01,  ...,  6.4787e-01,
          -1.3423e+00,  2.5104e-02],
         [-5.2108e-01, -6.9235e-01,  1.3100e+00,  ..., -7.3710e-01,
          -1.9010e-01, -4.3106e-01],
         ...,
         [ 6.3304e-01,  1.4327e+00,  3.6466e+00,  ...,  9.4099e-02,
          -3.6146e-01, -5.8223e-01],
         [ 5.9848e-01, -3.3681e-01,  3.1169e-01,  ...,  1.3572e+00,
          -1.1452e-01, -8.9035e-02],
         [ 6.9715e-01,  1.6658e-01,  1.8263e+00,  ...,  1.1207e+00,
          -3.6839e-01,  1.1169e+00]],

        [[-5.3258e-01,  1.4811e-01,  3.3389e-01,  ..., -6.5413e-02,
          -1.8645e+00,  7.6993e-01],
         [-1.2909e+00,  1.5740e+00,  7.9410e-01,  ...,  2.0888e+00,
          -2.3740e-01,  3.2583e-01],
         [ 9.9931e-01, -1.2265e+00,  1.9994e+00,  ...,  9.1879e-01,
         

# Decode阶段

In [27]:
class DecoderLayer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args

        # 1. masked self-attention
        self.self_attention = MultiHeadAttention(args, is_causal=True)
        self.self_attention_norm = LayerNorm(args.dim)

        # 2. cross-attention
        self.cross_attention = MultiHeadAttention(args, is_causal=False)
        self.cross_attention_norm = LayerNorm(args.dim)

        # 3. feed forward
        self.ffn = FNN(FNNArgs(dim=args.dim, hidden_dim=args.hidden_dim, dropout=args.dropout))
        self.ffn_norm = LayerNorm(args.dim)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor:
        '''
        x: [batch_size, tgt_len, dim]
        encoder_output: [batch_size, src_len, dim]
        '''
        # masked self-attention
        h = self.self_attention(x, x, x)
        h = h + x
        h = self.self_attention_norm(h)

        # cross-attention
        c = self.cross_attention(h, encoder_output, encoder_output)
        c = c + h
        c = self.cross_attention_norm(c)

        # feed forward
        out = self.ffn(c)
        out = out + c
        out = self.ffn_norm(out)

        return out


class Decoder(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.position_encoding = PositionEncoding(args)
        self.layers = nn.ModuleList([DecoderLayer(args) for _ in range(args.n_layers)])

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor:
        '''
        x: [batch_size, tgt_len, dim]
        encoder_output: [batch_size, src_len, dim]
        '''
        x = self.position_encoding(x)
        for layer in self.layers:
            x = layer(x, encoder_output)
        return x

In [28]:
args = ModelArgs(
    n_heads=8,
    dim=768,
    hidden_dim=768 * 4,
    dropout=0.1,
    max_seq_len=512,
    n_layers=6
)

batch_size = 10
src_len = 512
tgt_len = 128

src_x = torch.randn(batch_size, src_len, args.dim)
tgt_x = torch.randn(batch_size, tgt_len, args.dim)

encoder = Encoder(args)
decoder = Decoder(args)

encoder_output = encoder(src_x)
decoder_output = decoder(tgt_x, encoder_output)

print("encoder output shape:", encoder_output.shape)
print("decoder output shape:", decoder_output.shape)

encoder output shape: torch.Size([10, 512, 768])
decoder output shape: torch.Size([10, 128, 768])
